# Run experiments

In [ ]:
# releaese; stable
import json
import os
import argparse
import subprocess
import numpy as np
from src.utils import myrepr

# Function to create and submit SLURM job scripts
def submit_slurm_job(script_path, args, folder, job_name, submit):
    outs_folder = os.path.join(folder, "slurm_outs")
    sh_folder = os.path.join(folder, "slurm_sh")
    os.makedirs(outs_folder, exist_ok=True)
    os.makedirs(sh_folder, exist_ok=True)
    slurm_script = os.path.join(sh_folder, f"{job_name}.sh")
    
    # Create SLURM job script
    with open(slurm_script, "w") as f:
        f.write("#!/bin/bash\n")
        f.write(f"#SBATCH --job-name={job_name}\n")
        f.write("#SBATCH --cpus-per-task=1\n")
        #f.write('#SBATCH --constraint="intel&cascadelake"\n')
        f.write("#SBATCH -N 1\n")
        f.write("#SBATCH --partition=batch\n")
        f.write("#SBATCH --mem=8GB\n")
        f.write("#SBATCH --time=6:00:00\n")
        f.write(f"#SBATCH --output={outs_folder}/{job_name}.out\n")
        f.write(f"python {script_path} ")
        # for arg in args:
        #     f.write(f"{arg} ")
        
        for arg, value in zip(args[0::2], args[1::2]):
            if value != "":
                f.write(f"    {arg} {value} \\\n")    
        
        f.write("\n")
    if submit:
        # Submit the job using sbatch and capture the output
        result = subprocess.run(["sbatch", slurm_script], capture_output=True, text=True)
        
        # Print the result of the job submission
        if result.returncode == 0:
            #print(f"Job {job_name} submitted successfully: {result.stdout.strip()}")
            print(f"{job_name}")
        else:
            print(f"Job {job_name} submission failed: {result.stderr.strip()}")



In [ ]:
# releaese; stable

# Function to properly format list arguments
def format_list_args(args):
    list_args = [
        "--stop_criteruim_params",
        "--collectable_metrics",
        "--computable_params",
        "--loadable_params",
        "--loadable_datasets"
    ]
    for i in range(len(args)):
        if args[i] in list_args:
            args[i + 1] = f'"{args[i + 1]}"'
    return args

# ---------------------------------------------------------------------------
# 3) Decide which parameters each experiment loops over
#    E.g. GD only depends on factor; RAC-LoRA variants also use rank; 
#         RC-LoRA uses factor, rank, prob.
# ---------------------------------------------------------------------------
def generate_param_combinations(exp_name, arrays_dict):
    """
    Given an experiment name (e.g. 'GD', 'RAC-LoRA_A', etc.) and
    the arrays_dict containing factor_ar, rank_ar, prob_ar,
    generate only the relevant combinations of parameters.
    """
    factor_ar = arrays_dict["factor_ar"]
    rank_ar   = arrays_dict["rank_ar"]
    prob_ar   = arrays_dict["prob_ar"]
    
    # We'll store each combination as a dict, e.g.:
    # { 'factor': 1.0, 'rank': 10, 'prob': 0.5 }
    # or for GD (which doesn't need rank or prob),
    # { 'factor': 1.0 } etc.
    combinations = []

    if exp_name == "GD":
        # Only iterate over factor
        for f in factor_ar:
            combinations.append({"factor": f})

    elif exp_name in ["RAC-LoRA_A", "RAC-LoRA_B", 'finite_RAC-LoRA_A-SGD', 'finite_RAC-LoRA_B-SGD']:
        # factor, rank
        for f in factor_ar:
            for r in rank_ar:
                combinations.append({"factor": f, "rank": r})

    elif exp_name in ["RC-LoRA", 'finite_RC-LoRA-SGD', 'finite_RC-LoRA-PAGE']:
        # factor, rank, prob
        for f in factor_ar:
            for r in rank_ar:
                for p in prob_ar:
                    combinations.append({"factor": f, "rank": r, "prob": p})
    else:
        raise ValueError(f"Unknown experiment name: {exp_name}")
    # Default: if unknown name, no combos (or return something else)
    return combinations


# ---------------------------------------------------------------------------
# 4) The main function to process job submissions
# ---------------------------------------------------------------------------
def process_job_submissions(arrays_dict, submit, exps_to_launch, json_config):
    # Load JSON config
    config = json.loads(json_config)
    
    # Base args
    base_args = config["base_config"]["args"]
    base_args = format_list_args(base_args)
    
    # Dummy parse, to keep the same structure as before
    parser = argparse.ArgumentParser(description="Submit SLURM jobs with JSON configuration")
    parser.add_argument("--folder", type=str, help="Custom folder path to use as the working directory", default="")
    args_namespace = parser.parse_args()
    
    # We'll just use the current directory as the working directory
    working_directory = os.getcwd()

    # Iterate over each configuration in the JSON
    for conf in config["configurations"]:
        exp_name = conf["name"]
        if exp_name not in exps_to_launch:
            continue
        
        # Additional arguments for this experiment
        exp_conf_args = format_list_args(conf["args"])
        
        # Generate only the parameter combos relevant to this experiment
        param_combos = generate_param_combinations(exp_name, arrays_dict)
        
        # If param_combos is empty, it means that experiment doesn't need loops
        # or the config name wasn't recognized.
        if not param_combos:
            # If you want to still run something, do so here.
            # Example: if you want "GD" to run with NO combos at all, 
            # you might want to at least do 1 run. 
            # But in this example we've covered "GD" above, so there's always something.
            continue
        
        # Grab some base items to embed in the job name
        loss_func = base_args[base_args.index("--loss_func") + 1]
        step_size = base_args[base_args.index("--step_size") + 1]
        
        # Submit each combination
        for combo in param_combos:
            # Build up dynamic CLI args from the combo
            dynamic_args = []
            if "factor" in combo:
                dynamic_args += ["--factor", str(combo["factor"])]
            if "rank" in combo:
                dynamic_args += ["--rank", str(combo["rank"])]
            if "prob" in combo:
                dynamic_args += ["--prob", str(combo["prob"])]
            
            # Combine with base + experiment-specific arguments
            all_conf_args = base_args + exp_conf_args + dynamic_args
            
            # Build job_name
            # E.g. "lin-reg_GD_sconst_f1.0" or "lin-reg_RAC-LoRA_A_sconst_f2.0_rank8" or ...
            job_name = f"{loss_func}_{exp_name}_s{step_size}"
            if "factor" in combo:
                job_name += f"_f{myrepr(combo['factor'])}"
            if "rank" in combo:
                job_name += f"_rank{myrepr(combo['rank'])}"
            if "prob" in combo:
                job_name += f"_p{myrepr(combo['prob'])}"
            
            submit_slurm_job(script_path=conf["program"], args=all_conf_args, folder=working_directory, job_name=job_name, submit=submit)

In [ ]:

# releaese; stable
json_config = '''
{
    "version": "0.2.0",
    "base_config": {
        "type": "debugpy",
        "request": "launch",
        "console": "integratedTerminal",
        "justMyCode": true,
        "args": [
            "--dataset", "synthetic_dense",
            "--loss_func", "lin-reg",
            "--regularizer_type", "non-cvx",
            "--num_samples", "100000",
            "--tol", "1e-16",
            "--dim", "4096",
            "--use_ray", "0",
            "--is_sparse_dataset", "0",
            "--print_status", "1",
            "--is_grad_comp_init", "0",
            "--PRINT_EVERY", "100",
            "--SAVE_EVERY", "10000000",
            "--NUM_LAUNCHES", "1",
            "--step_size", "const",
            "--stop_criteruim_params", "['iters', 'sqnorm', 'epochs']",
            "--loadable_datasets", "['X_ft', 'y_ft', 'c_ft', 'A_ft', 'b_ft']",
            "--computable_params", "[]",
            "--num_workers", "1",
            "--loadable_params", "['L_0_pre', 'L_0_ft','x_star_pre', 'f_star_pre','num_samples_pre','num_samples_ft','la_pre','la_ft']",
            "--collectable_metrics", "['iters', 'sqnorm', 'epochs']"
        ]
    },
    "configurations": [
        {
            "name": "GD",
            "program": "GD.py",
            "args": [
                "--alg_name", "GD",
                "--exp_name", "GD",
                "--seed", "1",  
                "--max_iters", "100"
            ]
        },
        {
            "name": "RAC-LoRA_A",
            "program": "RAC-LoRA_A.py",
            "args": [
                "--alg_name", "RAC-LoRA_A",
                "--exp_name", "RAC-LoRA_A",
                "--seed", "1",
                "--max_iters", "4000"
            ]
        },
        {
            "name": "RAC-LoRA_B",
            "program": "RAC-LoRA_B.py",
            "args": [
                "--alg_name", "RAC-LoRA_B",
                "--exp_name", "RAC-LoRA_B",
                "--seed", "2",
                "--max_iters", "4000"
            ]
        },
        {
            "name": "RC-LoRA",
            "program": "RC-LoRA.py",
            "args": [
                "--alg_name", "RC-LoRA",
                "--exp_name", "RC-LoRA",
                "--seed", "3",
                "--max_iters", "4000"
            ]
        },
        {
            "name": "finite_RC-LoRA-SGD",
            "program": "RC-LoRA.py",
            "args": [
                "--alg_name", "RC-LoRA",
                "--exp_name", "finite_RC-LoRA-SGD",
                "--seed", "90",
                "--max_iters", "100000",
                "--max_epochs", "300",
                "--batchsize", "100"
            ]
        },
        {
            "name": "finite_RAC-LoRA_A-SGD",
            "program": "RC-LoRA.py",
            "args": [
                "--alg_name", "RAC-LoRA_A",
                "--exp_name", "finite_RAC-LoRA_A-SGD",
                "--seed", "110",
                "--max_iters", "100000",
                "--max_epochs", "300",
                "--batchsize", "100"
            ]
        },
        {
            "name": "finite_RAC-LoRA_B-SGD",
            "program": "RC-LoRA.py",
            "args": [
                "--alg_name", "RAC-LoRA_B",
                "--exp_name", "finite_RAC-LoRA_B-SGD",
                "--seed", "130",
                "--max_iters", "100000",
                "--max_epochs", "300",
                "--batchsize", "100"
            ]
        },
        {
            "name": "finite_RC-LoRA-PAGE",
            "program": "RC-LoRA.py",
            "args": [
                "--alg_name", "RC-LoRA",
                "--exp_name", "finite_RC-LoRA-PAGE",
                "--seed", "150",
                "--max_iters", "100000",
                "--max_epochs", "300",
                "--batchsize", "100"
            ]
        }
    ]
}
'''

submit = 1

#exps_to_launch = ['']
#exps_to_launch = ['']
#exps_to_launch = ['']

exps_to_launch = ['finite_RC-LoRA-SGD', 'finite_RAC-LoRA_A-SGD', 'finite_RAC-LoRA_B-SGD', 'finite_RC-LoRA-PAGE']
#exps_to_launch = ['finite_RC-LoRA-SGD', 'finite_RAC-LoRA_A-SGD', 'finite_RAC-LoRA_B-SGD']

#exps_to_launch = ['finite_RC-LoRA-SGD']
arrays_dict = {
#"factor_ar": np.array([1,2], dtype=float),
    "factor_ar": np.array([0.0625, 0.03125, 0.015625, 0.0078125, 0.125, 0.25, 0.5, 1,2,4], dtype=float),
    #"factor_ar": np.array([1,2], dtype=float),
    "rank_ar": np.array([2], dtype=int),
    #"prob_ar": np.array([0.1, 0.9], dtype=float),
    "prob_ar": np.array([0.5], dtype=float)
}

process_job_submissions(arrays_dict, submit, exps_to_launch, json_config)


In [ ]:
!rm -r slurm_sh/*.sh

In [ ]:
!rm -r slurm_outs/*.out


In [ ]:
!rm -r logs/*

In [ ]:
%%bash
sacct --format=JobID,JobName%50,Start,End,Elapsed,State --starttime=now-1hour

In [ ]:
%%bash
sacct --starttime=2025-01-18 --endtime=2024-01-18 --format=JobID,JobName%50,Start,End,Elapsed,State --starttime=now-1hour